# Taller: Ciclo de Vida de un Modelo de ML en Seguros / Suscripción

En este notebook construiremos, evaluaremos y serializaremos un modelo de retención de cartera.

### Objetivo:
1. Cargar y preparar la base de asegurados/clientes.
2. Armar un pipeline de preprocesamiento robusto (`ColumnTransformer`).
3. Entrenar y calibrar un clasificador.
4. Medir métricas actuariales clave (**ROC-AUC** y **Brier Score**).
5. Exportar el artefacto `modelo_retencion.joblib` para ponerlo en producción en GitHub Actions.

In [ ]:
# 1. Librerías requeridas
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    classification_report,
    roc_curve
)

# 2. Cargar datos de entrenamiento desde el repositorio
# (Asegúrate de cambiar TU_USUARIO por el usuario dueño del repo si es necesario)
URL_DATOS = "https://raw.githubusercontent.com/ClauJuarez/taller-mlops-actuaria/main/data/telco_train_alumnos.csv"
df = pd.read_csv(URL_DATOS)

print(f"Dimensiones: {df.shape}")
df.head()

### Definición de Variables y Target
Separamos las covariables ($X$) de la variable objetivo ($y$). `customerID` es un identificador sin valor predictivo.

In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

print(f"Variables numéricas: {num_cols}")
print(f"Variables categóricas: {cat_cols}")

### TODO 1: Partición Train / Test Estratificada
Divide los datos en 80% entrenamiento y 20% validación usando `stratify=y`.

In [ ]:
# TODO 1: Completa los parámetros test_size y stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20,      # <--- Completa
    stratify=y,          # <--- Completa
    random_state=42
)

print(f"Entrenamiento: {len(X_train)} filas | Tasa Churn: {y_train.mean():.2%}")
print(f"Validación:    {len(X_test)} filas | Tasa Churn: {y_test.mean():.2%}")

### Construcción del Pipeline de Preprocesamiento
Escalamos las variables numéricas y codificamos las variables categóricas con OneHotEncoder.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols)
    ]
)

### TODO 2: Modelo y Pipeline Completo
Configura el clasificador con balanceo de clases y compón el `Pipeline`.

In [ ]:
# TODO 2: Configura el clasificador con class_weight="balanced"
modelo_base = LogisticRegression(
    max_iter=1000, 
    class_weight="balanced",  # <--- Discutir impacto
    random_state=42
)

pipeline_completo = Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("clasificador", modelo_base)
])

pipeline_completo.fit(X_train, y_train)
print("¡Modelo entrenado exitosamente!")

### TODO 3: Evaluación de Métricas de Negocio
Calcula el **ROC-AUC** y el **Brier Score** sobre el conjunto de validación (`X_test`, `y_test`).

In [ ]:
# Extraer probabilidades estimadas
y_prob_test = pipeline_completo.predict_proba(X_test)[:, 1]

# TODO 3: Calcular métricas base para el monitoreo
auc_score = roc_auc_score(y_test, y_prob_test)
brier_score = brier_score_loss(y_test, y_prob_test)

print("--- RESULTADOS DE VALIDACIÓN ---")
print(f"ROC-AUC Base:    {auc_score:.4f}  (Línea base para producción)")
print(f"Brier Score:     {brier_score:.4f} (Idealmente cercano a 0)")

y_pred_default = (y_prob_test >= 0.50).astype(int)
print("\nReporte de Clasificación (Corte 0.50):")
print(classification_report(y_test, y_pred_default))

### TODO 4: Exportación del Modelo Serializado
Guarda el pipeline completo en disco como `modelo_retencion.joblib`.

In [ ]:
# TODO 4: Serializar el objeto pipeline_completo
NOMBRE_ARTEFACTO = "modelo_retencion.joblib"
joblib.dump(pipeline_completo, NOMBRE_ARTEFACTO)

print(f"Artefacto guardado: '{NOMBRE_ARTEFACTO}'")
print("Descárgalo y súbelo a la carpeta models/ de tu repositorio de GitHub.")